# Databricks EU Workspace - MLOps Smoke Test

Use this notebook to verify that the main MLOps components are working in the EU Databricks workspace.

It checks:
- Python / scikit-learn availability
- MLflow experiment tracking
- Metric and parameter logging
- Artifact logging
- Model logging
- Optional Unity Catalog model registration


In [ ]:
import sys
import mlflow
import sklearn
import pandas as pd

print('Python version:', sys.version)
print('MLflow version:', mlflow.__version__)
print('scikit-learn version:', sklearn.__version__)
print('pandas version:', pd.__version__)
print('Basic libraries loaded successfully.')

## 1. Create / use an MLflow experiment

This uses a workspace experiment. Change the path if required.

In [ ]:
experiment_name = '/Shared/eu_mlops_smoke_test'
mlflow.set_experiment(experiment_name)
print('Using experiment:', experiment_name)

## 2. Train a small sample model and log it to MLflow

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = LogisticRegression(max_iter=500)
model.fit(X_train, y_train)

pred = model.predict(X_test)

accuracy = accuracy_score(y_test, pred)
precision = precision_score(y_test, pred, average='weighted', zero_division=0)
recall = recall_score(y_test, pred, average='weighted', zero_division=0)
f1 = f1_score(y_test, pred, average='weighted', zero_division=0)

print('Accuracy :', accuracy)
print('Precision:', precision)
print('Recall   :', recall)
print('F1       :', f1)

In [ ]:
import tempfile
from pathlib import Path

with mlflow.start_run(run_name='eu_mlops_smoke_test') as run:
    mlflow.log_param('model_type', 'LogisticRegression')
    mlflow.log_param('max_iter', 500)

    mlflow.log_metric('accuracy', accuracy)
    mlflow.log_metric('precision', precision)
    mlflow.log_metric('recall', recall)
    mlflow.log_metric('f1', f1)

    with tempfile.TemporaryDirectory() as tmpdir:
        artifact_file = Path(tmpdir) / 'smoke_test.txt'
        artifact_file.write_text('EU Databricks MLOps smoke test artifact created successfully.')
        mlflow.log_artifact(str(artifact_file))

    mlflow.sklearn.log_model(model, artifact_path='model')

    run_id = run.info.run_id
    print('Run ID:', run_id)
    print('MLflow tracking + metrics + artifact + model logging completed successfully.')

## 3. Load the logged model back

If this succeeds, the logged model can be retrieved from MLflow.

In [ ]:
logged_model_uri = f'runs:/{run_id}/model'
loaded_model = mlflow.sklearn.load_model(logged_model_uri)
loaded_pred = loaded_model.predict(X_test)

print('Loaded model accuracy:', accuracy_score(y_test, loaded_pred))
print('Model load test successful.')

## 4. Optional - test Unity Catalog model registration

Run this only if your EU workspace has Unity Catalog permissions and you know the target catalog/schema.

Replace the sample name below with your approved catalog and schema.

In [ ]:
# OPTIONAL CELL
# Uncomment after replacing with a valid UC catalog.schema.model_name.

# mlflow.set_registry_uri('databricks-uc')
# uc_model_name = 'YOUR_CATALOG.YOUR_SCHEMA.eu_mlops_smoke_test_model'
# result = mlflow.register_model(
#     model_uri=f'runs:/{run_id}/model',
#     name=uc_model_name
# )
# print('Registered model version:', result.version)

## How to interpret the result

- If cells 1-3 work: core MLflow experiment tracking and model logging are working.
- If the model loads back successfully: artifact/model storage is working.
- If the optional Unity Catalog registration works: model registry permissions are also working.
- If UC registration fails but earlier cells work: MLOps is partially available, and the issue is likely UC catalog/schema/model permissions rather than MLflow itself.
